# perfcost quick start

Trains two small scikit-learn models on the digits dataset and logs them to the dashboard, the second one live (slowed down so you can watch it). Needs `pip install scikit-learn`.

In [ ]:
import time

import perfcost
from sklearn.datasets import load_digits
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC

X, y = load_digits(return_X_y=True)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3, stratify=y, random_state=0)
DATASET = {"name": "Digits", "modality": "image (8x8)", "n_train": len(X_tr), "n_test": len(X_te), "classes": 10}
LABELS = [str(i) for i in range(10)]

In [ ]:
def evaluate(name, model, slow=0.0):
    t0 = time.perf_counter()
    model.fit(X_tr, y_tr)
    train_s = time.perf_counter() - t0
    run = perfcost.start(name, dataset=DATASET, model={"name": type(model).__name__, "strategy": "trained"},
                         labels=LABELS, expected_n=len(X_te))
    run.log_cost(kind="training", wall_time_s=train_s, energy_kwh=train_s * 50 / 3.6e6)   # ~50 W, illustrative
    for x, t in zip(X_te, y_te):
        t0 = time.perf_counter()
        p = model.predict_proba([x])[0]
        ms = (time.perf_counter() - t0) * 1000
        run.log(y_true=str(t), y_pred=str(p.argmax()), confidence=float(p.max()), latency_ms=ms)
        time.sleep(slow)
    run.log_cost(kind="inference", mode="online", n=len(X_te),
                 energy_kwh=len(X_te) * 0.001 * 50 / 3.6e6)              # illustrative
    run.finish()
    return run

evaluate("digits / logistic regression", LogisticRegression(max_iter=2000))

In [ ]:
evaluate("digits / RBF SVM (live)", SVC(probability=True), slow=0.05)   # watch it in the dashboard

In [ ]:
# perfcost.stop_server()   # when you are done